In [1]:
from pathlib import Path
import os
import joblib
from tqdm import tqdm
import ast

import pandas as pd
import numpy as np
import scipy.sparse as sp
from sqlalchemy import create_engine, text

import torch
import faiss
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


BASE_DIR = Path.cwd().parent
ABSTRACTS_PATH = BASE_DIR / "data" / "arxiv_metadata.csv"
OUTPUT_EMBEDDINGS_PATH = BASE_DIR / "data" / "BERT_embeddings" / "embeddings.npy"
OUTPUT_INDEX_PATH = BASE_DIR / "data" / "BERT_embeddings" / "faiss_index.index"

/Users/salirafi/Documents/Personal Project/Abstract Recommender/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [14]:
abstracts_df = pd.read_csv(ABSTRACTS_PATH)
print(f"Total papers to embed: {len(abstracts_df)}")
abstracts_df

Total papers to embed: 381344


/var/folders/kz/dlw3ql7s3ds1_72hwlw7_t540000gn/T/ipykernel_75492/2732779598.py:1: DtypeWarning: Columns (0: paper_id) have mixed types. Specify dtype option on import or set low_memory=False.
  abstracts_df = pd.read_csv(ABSTRACTS_PATH)


,id,paper_id,title,abstract,authors,categories,update_year
0,0,704.0009,"The Spitzer c2d Survey of Large, Nearby, Inste...",We discuss the results from the combined IRA...,"[""Harvey Paul"", ""Merin Bruno"", ""Huard Tracy L....","[""astro-ph""]",2010
1,1,704.0017,Spectroscopic Observations of the Intermediate...,Results from spectroscopic observations of t...,"[""Mhlahlo Nceba"", ""Buckley David H."", ""Dhillon...","[""astro-ph""]",2009
2,2,704.0023,ALMA as the ideal probe of the solar chromosphere,"The very nature of the solar chromosphere, i...","[""Loukitcheva M. A."", ""Solanki S. K."", ""White ...","[""astro-ph""]",2009
3,3,704.0044,Astrophysical gyrokinetics: kinetic and fluid ...,We present a theoretical framework for plasm...,"[""Schekochihin A. A."", ""Cowley S. C."", ""Dorlan...","[""astro-ph"", ""nlin.CD"", ""physics.plasm-ph"", ""p...",2015
4,4,704.0048,Inference on white dwarf binary systems using ...,We report on the analysis of selected single...,"[""Stroeer Alexander"", ""Veitch John"", ""Roever C...","[""gr-qc"", ""astro-ph""]",2008
...,...,...,...,...,...,...,...
381339,381339,quant-ph/9903043,A Possible Anisotropy in Blackbody Radiation V...,A non-local gauge symmetry of a complex scal...,"[""Dastidar T K Rai""]","[""quant-ph"", ""astro-ph"", ""hep-th""]",2007
381340,381340,quant-ph/9903053,Father Time. I. Does the Cosmic Microwave Back...,The existence of a non-thermodynamic arrow o...,"[""Dastidar T K Rai""]","[""quant-ph"", ""astro-ph"", ""hep-th""]",2009
381341,381341,quant-ph/9907088,On Bures fidelity of displaced squeezed therma...,Fidelity plays a key role in quantum informa...,"[""Wang Xiang-Bin"", ""Oh C. H."", ""Kwek L. C.""]","[""quant-ph"", ""astro-ph""]",2008
381342,381342,solv-int/9404002,Dynamical Systems Accepting the Normal Shift,Newtonian dynamical systems accepting the no...,"[""Sharipov R. A.""]","[""solv-int"", ""alg-geom"", ""astro-ph"", ""gr-qc"", ...",2008


In [15]:
model_name = "sentence-transformers/all-distilroberta-v1"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu") # I'm running in macos
model = model.to(device)
model.eval() # inference mode

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 13990.97it/s]
RobertaModel LOAD REPORT from: sentence-transformers/all-distilroberta-v1
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


RobertaModel(
  (embeddings): RobertaEmbeddings(
    (word_embeddings): Embedding(50265, 768, padding_idx=1)
    (token_type_embeddings): Embedding(1, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
    (position_embeddings): Embedding(514, 768, padding_idx=1)
  )
  (encoder): RobertaEncoder(
    (layer): ModuleList(
      (0-5): 6 x RobertaLayer(
        (attention): RobertaAttention(
          (self): RobertaSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): RobertaSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
            (dropout)

In [4]:
index = faiss.read_index(str(OUTPUT_INDEX_PATH))

In [16]:
def embed(text):

    model.eval()
    encoded = tokenizer(
        [text],
        padding=True,
        truncation=True,
        max_length=512,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        output = model(**encoded)

    token_embeddings = output.last_hidden_state
    attention_mask = encoded.attention_mask
    mask_expanded = attention_mask.unsqueeze(-1).float()
    sum_embeddings = torch.sum(token_embeddings * mask_expanded, dim=1)
    sum_mask = mask_expanded.sum(dim=1).clamp(min=1e-9)
    mean_embeddings = sum_embeddings / sum_mask
    query_vector = F.normalize(mean_embeddings, p=2, dim=1).cpu().numpy().astype("float32")

    return query_vector

In [17]:
def recommend(text: str, top_k: int = 50):

    query_vector = embed(text)
    scores, indices = index.search(query_vector, top_k)

    results = []
    for idx, score in zip(indices[0], scores[0]):
        row = df.iloc[idx]
        results.append({
            "title": row["title"],
            "categories": row["categories"],
            "update_year": row["update_year"],
            "score": round(float(score), 4),
        })
    return pd.DataFrame(results)

In [12]:
query = """
We investigate the rotational dynamics of Earth's liquid core using VLBI 
observations. Our analysis reveals two distinct nutation periods near -430 
solar days, suggesting a two-layer liquid core structure.
"""

results = recommend(query, top_k=10)
print(results.to_string())

: 

#### The Precision@K is based on whether the recommended papers are within the same category or not, which may be a bit biased because the categories themselves are not inherently special. For this, we need a more semantical way to compare between abstracts which is not covered in the project.

In [ ]:


def parse_categories(cat_str):
    try:
        return set(ast.literal_eval(cat_str))
    except:
        return set()



def precision_at_k(query_idx: int, top_k: int = 5, df: pd.DataFrame = abstracts_df) -> float:
    query_row = df.iloc[query_idx]
    query_abstract = query_row["abstract"]
    query_cats = parse_categories(query_row["categories"])

    query_vector = embed(query_abstract)
    scores, indices = index.search(query_vector, top_k + 1)

    retrieved = [idx for idx in indices[0] if idx != query_idx][:top_k]

    matches = sum(
        1 for idx in retrieved
        if parse_categories(df.iloc[idx]["categories"]) & query_cats  # set intersection
    )

    return matches / top_k

In [ ]:
# evaluate on a random sample of 1000 papers
sample_size = 1000
np.random.seed(42)
sample_indices = np.random.choice(len(abstracts_df), size=sample_size, replace=False)

scores = []
for idx in tqdm(sample_indices):
    scores.append(precision_at_k(idx, top_k=5))

mean_precision = np.mean(scores)
print(f"Mean Precision@5: {mean_precision:.4f}")

  0%|          | 0/1000 [00:00<?, ?it/s]

: 

In [ ]:
VECTORIZER_PATH = BASE_DIR / "data" / "tfidf_vectorizer.joblib"
MATRIX_PATH = BASE_DIR / "data" / "tfidf_matrix.joblib"
vectorizer = joblib.load(VECTORIZER_PATH)
tfidf_matrix = joblib.load(MATRIX_PATH)
print(f"TF-IDF matrix shape: {tfidf_matrix.shape}")

def precision_at_k_tfidf(query_idx: int, top_k: int = 5) -> float:
    query_vector = tfidf_matrix[query_idx]
    query_cats = parse_categories(abstracts_df.iloc[query_idx]["categories"])

    # compute cosine similarity between query and all papers explicitly
    scores = cosine_similarity(query_vector, tfidf_matrix).flatten()

    
    top_indices = np.argsort(scores)[::-1]
    retrieved = [idx for idx in top_indices if idx != query_idx][:top_k]

    matches = sum(
        1 for idx in retrieved
        if parse_categories(abstracts_df.iloc[idx]["categories"]) & query_cats
    )

    return matches / top_k

# evaluate tf-idf on the same sample
tfidf_scores = []
for idx in tqdm(sample_indices):
    tfidf_scores.append(precision_at_k_tfidf(idx, top_k=5))

mean_tfidf_precision = np.mean(tfidf_scores)
print(f"TF-IDF Mean Precision@5:     {mean_tfidf_precision:.4f}")
print(f"Embedding Mean Precision@5:  0.7580")
print(f"Improvement:                 +{0.7580 - mean_tfidf_precision:.4f}")

TF-IDF matrix shape: (381344, 10000)


100%|██████████| 1000/1000 [07:11<00:00,  2.32it/s]

TF-IDF Mean Precision@5:     0.7218
Embedding Mean Precision@5:  0.7580
Improvement:                 +0.0362
